# 05 - Explorador interactivo de figuras del paper

Este notebook te permite **elegir Red, Radio y Steps** con menús desplegables y reproduce,
paso a paso, cada figura del paper, explicando qué se hizo para obtenerla.

**Figuras del paper que se replican aquí:**
1. `All_SlopesLinearFit_vs_InstantOfTime_with_TrafficIntensity.png` → pendientes vs hora del día (función `run_hours_vs_slope`).
2. `AllTopology_SlopeLinerFit_vs_P95TrafficIntensity_ModelDGN1207.png` → Slope vs P95 para cada índice topológico (función `run_median_slopes_vs_per95`).
3. `CC_vs_P95TrafficIntensity_allModels.png` → comparación CC vs P95 entre modelos (función `run_three_slopes_vs_per95` / scatter de índices).

Para ejecutar los widgets necesitas `ipywidgets`:
`pip install ipywidgets` y luego `jupyter nbextension enable --py widgetsnbextension` (o usar JupyterLab).

In [ ]:
import sys, os
# Cambiar al directorio raiz del proyecto (padre de notebooks/)
ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__) if '__file__' in dir() else os.getcwd(), '..'))
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

from functions.RutasDeArchivos import (
    Datos_computacionales,
    datos_combinaciones,
    REDES,
    RADIOS,
    STEPS,
)
from src.visualization.MedianSlopesVsPer95 import run_median_slopes_vs_per95
from src.visualization.MedianSlopesVsR2 import run_median_slopes_vs_r2
from src.visualization.ThreeSlopesVsPer95 import run_three_slopes_vs_per95
from src.visualization.Hours_vs_Slope_TrafficScale import run_hours_vs_slope


## Paso 0: Selección de parámetros

Elige la **Red**, el **Radio** y los **Steps**. Estos definen qué datos observacionales
se usan para todas las figuras siguientes.

- **Red**: `N505` (SimpleNet) o `N1207` (ComplexNet).
- **Radio**: parámetro de muestreo espacial de las imágenes de tráfico.
- **Steps**: paso temporal entre muestras de tráfico.

In [ ]:
red_sel = widgets.Dropdown(options=REDES, value=REDES[0], description='Red:')
radio_sel = widgets.Dropdown(options=RADIOS, value=RADIOS[0], description='Radio:')
steps_sel = widgets.Dropdown(options=STEPS, value=STEPS[0], description='Steps:')
params_box = widgets.VBox([red_sel, radio_sel, steps_sel])
display(params_box)

In [ ]:
# Cargar datos según la selección
def cargar_datos(red, radio, steps):
    datos_comp = pd.read_csv(Datos_computacionales[REDES.index(red)])
    obs_path = datos_combinaciones[red]['mean'][(radio, steps)]
    if not os.path.exists(obs_path):
        raise FileNotFoundError(
            f'No existe {obs_path}.\n'
            'Prueba otra combinación de Radio/Steps o genera los datos primero.'
        )
    datos_obs = pd.read_csv(obs_path)
    return datos_comp, datos_obs

red = red_sel.value
radio = radio_sel.value
steps = steps_sel.value
datos_comp, datos_obs = cargar_datos(red, radio, steps)

display(Markdown(f"**Red seleccionada:** `{red}`  |  **Radio:** `{radio}`  |  **Steps:** `{steps}`"))
print(f'Obs : {datos_obs.shape}  <- {datos_combinaciones[red]["mean"][(radio, steps)]}')
print(f'Comp: {datos_comp.shape}')

## Paso 1: Figura del paper 1 — Pendientes vs hora del día

**Imagen del paper:** `All_SlopesLinearFit_vs_InstantOfTime_with_TrafficIntensity.png`

**Qué se hizo paso a paso:**
1. Se recorren los 24×4 = 96 instantes del día (cada 15 min).
2. Para cada instante se extraen las columnas de tráfico observado, se reescalan a [0,255] y se calcula el promedio por nodo.
3. Se calcula el **P95** de esos promedios → nivel de intensidad de tráfico de la ciudad (barra de color gris a la derecha).
4. Para cada índice topológico (`DiBC, BC, DiCC, CC, DiDC, DC`) se normaliza la centralidad y se divide en *n_boxes* intervalos ordenados (15 para BC, 20 para CC, 3 para DC).
5. Se ajusta una recta (`scipy.stats.linregress`) a las **medianas** tráfico vs centralidad en cada intervalo. La **pendiente** de esa recta es la correlación topología–tráfico en ese instante.
6. Se grafican las 6 pendientes vs el minuto del día, con la barra de tráfico de fondo.

La función utilizada es `run_hours_vs_slope` (ya corregida para no achatar la imagen).

In [ ]:
run_hours_vs_slope(datos_comp, datos_obs, red)

## Paso 2: Figura del paper 2 — Slope vs P95 por índice topológico

**Imagen del paper:** `AllTopology_SlopeLinerFit_vs_P95TrafficIntensity_ModelDGN1207.png`

**Qué se hizo paso a paso:**
1. Para cada instante del día se repite el cálculo de la pendiente (misma idea del Paso 1) para **un solo** índice topológico.
2. Se empareja cada pendiente con el **P95** de tráfico de ese instante.
3. Se aplica un **umbral α** (threshold) para filtrar los instantes de bajo tráfico y aislar el régimen de congestión.
4. Se ajusta una recta (R²) a los puntos filtrados: si la pendiente es positiva y R² alto, la topología predice bien el tráfico.

Aquí se genera para todos los índices topológicos de la red elegida. El modelo óptimo del paper usa `DiCC` con el umbral de la red.

In [ ]:
OPTIMAL_THRESHOLDS = {"N505": 0.329, "N1207": 0.368}
for idx_name in ['DiCC', 'CC', 'DiBC', 'BC', 'DiDC', 'DC']:
    display(Markdown(f'---\n**Índice topológico:** `{idx_name}`'))
    run_median_slopes_vs_per95(
        datos_comp=datos_comp,
        datos_obs=datos_obs,
        label=red,
        topology_index=idx_name,
        threshold=OPTIMAL_THRESHOLDS[red],
    )

## Paso 3: Figura del paper 3 — Comparación CC vs P95 entre modelos

**Imagen del paper:** `CC_vs_P95TrafficIntensity_allModels.png`

**Qué se hizo paso a paso:**
1. Se superponen varios índices topológicos (BC, CC, DC) en un solo gráfico Slope vs P95.
2. Cada uno lleva su propio umbral α y su ajuste lineal.
3. Esto permite comparar, para todos los modelos/topologías, cómo varía la correlación respecto a la intensidad de tráfico.

La función `run_three_slopes_vs_per95` replica esta superposición.

In [ ]:
THREE_SLOPES_THRESHOLDS = {"N505": [0.25, 0.332, 0.319], "N1207": [0.38, 0.328, 0.305]}
run_three_slopes_vs_per95(
    datos_comp=datos_comp,
    datos_obs=datos_obs,
    label=red,
    topology_indices=['BC', 'CC', 'DC'],
    thresholds=THREE_SLOPES_THRESHOLDS[red],
)

## Resumen

Con la combinación **Red / Radio / Steps** elegida arriba replicaste las 3 figuras principales del paper:

| # | Figura en el paper | Función | Qué muestra |
|---|--------------------|---------|-------------|
| 1 | `..._InstantOfTime_with_TrafficIntensity.png` | `run_hours_vs_slope` | Evolución 24h de la correlación topología–tráfico |
| 2 | `..._ModelDGN1207.png` | `run_median_slopes_vs_per95` | Slope vs P95 por índice topológico (umbral α) |
| 3 | `CC_vs_P95TrafficIntensity_allModels.png` | `run_three_slopes_vs_per95` | Superposición BC/CC/DC para comparar modelos |

Para regenerar todas las combinaciones en disco (solo `Hours_vs_Slope`, ya corregido) ejecuta:
```bash
python src/visualization/run_all_combinations.py
```